In [2]:
"""
Statistical Exploration and K-Means Clustering Pipeline
Libraries: Pandas, NumPy, Scikit-Learn
"""
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
# ==========================================
# 1. SAMPLE DATA GENERATOR
# ==========================================
def generate_customer_data(n_samples: int = 300) -> pd.DataFrame:
    """Generates synthetic customer behavioral data with distinct latent clusters."""
    np.random.seed(42)
    # Segment 1: Young, Low Income, High Spenders (e.g., Students / Early Career)
    c1_age = np.random.normal(22, 3, n_samples // 3)
    c1_income = np.random.normal(30000, 5000, n_samples // 3)
    c1_spend = np.random.normal(80, 8, n_samples // 3)
    # Segment 2: Middle-Aged, High Income, High Spenders (e.g., Affluent Loyalists)
    c2_age = np.random.normal(45, 6, n_samples // 3)
    c2_income = np.random.normal(120000, 15000, n_samples // 3)
    c2_spend = np.random.normal(85, 7, n_samples // 3)
    # Segment 3: Older, Moderate Income, Low Spenders (e.g., Conservative Savers)
    c3_age = np.random.normal(58, 5, n_samples // 3)
    c3_income = np.random.normal(65000, 8000, n_samples // 3)
    c3_spend = np.random.normal(25, 6, n_samples // 3)
    df = pd.DataFrame({
        "customer_id": np.arange(1001, 1001 + n_samples),
        "age": np.concatenate([c1_age, c2_age, c3_age]).round(1),
        "annual_income": np.concatenate([c1_income, c2_income, c3_income]).round(2),
        "spending_score": np.concatenate([c1_spend, c2_spend, c3_spend]).round(1)
    })
    return df
# ==========================================
# 2. STATISTICAL PROFILING ENGINE
# ==========================================
class StatisticalExplorer:
    """Calculates distributional properties, skewness, and correlation."""
    def __init__(self, df: pd.DataFrame, feature_cols: list[str]):
        self.df = df
        self.features = feature_cols
    def profile_distributions(self) -> pd.DataFrame:
        """Computes mean, std, variance, and skewness for each feature."""
        stats = pd.DataFrame({
            "Mean": self.df[self.features].mean(),
            "Median": self.df[self.features].median(),
            "Std Dev": self.df[self.features].std(),
            "Variance": self.df[self.features].var(),
            "Skewness": self.df[self.features].skew()
        }).round(3)
        return stats
    def compute_correlation(self) -> pd.DataFrame:
        """Returns feature correlation matrix."""
        return self.df[self.features].corr().round(3)
# ==========================================
# 3. K-MEANS CLUSTERING ENGINE
# ==========================================
class CustomerClusterer:
    """Standardizes features, determines optimal k, and executes K-Means clustering."""
    def __init__(self, df: pd.DataFrame, feature_cols: list[str]):
        self.df = df.copy()
        self.features = feature_cols
        self.scaler = StandardScaler()
        self.scaled_data = self.scaler.fit_transform(self.df[self.features])
    def evaluate_optimal_k(self, k_range: range = range(2, 7)) -> pd.DataFrame:
        """Calculates Inertia (WCSS) and Silhouette Scores for a range of k."""
        results = []
        for k in k_range:
            kmeans = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
            cluster_labels = kmeans.fit_predict(self.scaled_data)
            inertia = kmeans.inertia_
            sil_score = silhouette_score(self.scaled_data, cluster_labels)
            results.append({"k": k, "Inertia (WCSS)": round(inertia, 2), "Silhouette Score": round(sil_score, 4)})
        return pd.DataFrame(results)
    def fit_clusters(self, n_clusters: int = 3) -> pd.DataFrame:
        """Fits final K-Means model and assigns cluster labels to the dataset."""
        kmeans = KMeans(n_clusters=n_clusters, init="k-means++", random_state=42, n_init=10)
        self.df["cluster"] = kmeans.fit_predict(self.scaled_data)
        return self.df
    def profile_clusters(self) -> pd.DataFrame:
        """Summarizes cluster characteristics across selected features."""
        summary = self.df.groupby("cluster")[self.features].agg(["count", "mean", "std"]).round(2)
        return summary
# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    print("[1] Generating Customer Dataset...")
    data = generate_customer_data(n_samples=300)
    features = ["age", "annual_income", "spending_score"]
    print(data.head(5))
    print("\n[2] Statistical Exploration & Distribution Profiling:")
    stats_engine = StatisticalExplorer(data, features)
    print(stats_engine.profile_distributions())
    print("\n[3] Evaluating Optimal Clusters (k=2 to 6):")
    clusterer = CustomerClusterer(data, features)
    k_eval = clusterer.evaluate_optimal_k()
    print(k_eval.to_string(index=False))
    print("\n[4] Fitting Final K-Means Model (k=3)...")
    clustered_df = clusterer.fit_clusters(n_clusters=3)
    print("\n[5] Cluster Behavioral Profiles:")
    print(clusterer.profile_clusters())

[1] Generating Customer Dataset...
   customer_id   age  annual_income  spending_score
0         1001  23.5       22923.15            82.9
1         1002  21.6       27896.77            84.5
2         1003  23.9       28286.43            88.7
3         1004  26.6       25988.61            88.4
4         1005  21.3       29193.57            69.0

[2] Statistical Exploration & Distribution Profiling:
                     Mean     Median    Std Dev      Variance  Skewness
age                41.816     45.100     15.828  2.505300e+02    -0.211
annual_income   71407.390  65575.935  38195.499  1.458896e+09     0.432
spending_score     63.694     77.400     27.392  7.503050e+02    -0.584

[3] Evaluating Optimal Clusters (k=2 to 6):
 k  Inertia (WCSS)  Silhouette Score
 2          423.85            0.5644
 3           68.10            0.7765
 4           56.05            0.6331
 5           48.24            0.4868
 6           41.95            0.3369

[4] Fitting Final K-Means Model (k=3)...

